# From Pixels to Predictions

Building a convolutional neural network from the arithmetic up, then training it to tell cats from dogs.

- **Task** — binary classification, `0 = cat`, `1 = dog`
- **Data** — Oxford-IIIT Pet, 7,349 photographs, 37 breeds
- **Model** — ResNet-18, pretrained on ImageNet, fine-tuned here
- **Runtime** — a few minutes end to end; the dataset downloads once (~800 MB)

**Contents**

1. What a computer actually sees
2. Images as tensors
3. One convolutional layer
4. The convolution equation
5. Labels: 0 is cat, 1 is dog
6. The model
7. Backpropagation, epochs, hyperparameters
8. The training loop
9. Predicting on an unseen cat

### Setup

- One import block, one random seed, one visual style for every figure
- Device picked automatically: Apple GPU (`mps`), NVIDIA (`cuda`), or `cpu`

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch, FancyBboxPatch
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import resnet18, ResNet18_Weights

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = ("mps"  if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available()
          else "cpu")
DATA = Path("data")

# --- one visual language for every figure in the notebook -------------------
INK, INK_SOFT, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
CHANNELS = [("R", "#d62d2d"), ("G", "#1f9e4a"), ("B", "#2a78d6")]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "text.color": INK, "font.size": 10,
    "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK_SOFT,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False,
})

def frame(ax, colour="#dedcd6", lw=1.0):
    """Strip the ticks, keep a light box on all four sides."""
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(colour)
        spine.set_linewidth(lw)


def tint(hex_colour, v):
    """Opaque blend of a hex colour towards white. v in [0, 1]."""
    r, g, b = to_rgb(hex_colour)
    a = 0.12 + 0.78 * v
    return (1 - a + a * r, 1 - a + a * g, 1 - a + a * b)

print(f"torch {torch.__version__}   ·   device: {DEVICE}")

In [ ]:
# Downloads on first run only (~800 MB), then reads from disk.
train_pool = OxfordIIITPet(DATA, split="trainval", download=True)
test_pool  = OxfordIIITPet(DATA, split="test", download=True)

print(f"trainval  {len(train_pool):>5,} images")
print(f"test      {len(test_pool):>5,} images")
print(f"breeds    {len(train_pool.classes)}  "
      f"({train_pool.classes[0]}, {train_pool.classes[1]}, ..., {train_pool.classes[-1]})")

## 1. What a computer actually sees

**Computer vision**

- Turning a picture into a decision
- In: light, sampled on a rectangular grid
- Out: a label, a box, a mask, a number

**The gap**

- You see fur, whiskers, two ears, a face
- The computer receives integers in the range 0–255
- No edges, no objects, no *cat* — those have to be learned

**The grid**

- Height × width pixels
- Three matrices stacked: red, green, blue
- One integer per channel, per pixel

In [ ]:
CAT = DATA / "oxford-iiit-pet" / "images" / "Abyssinian_119.jpg"
N, ZY, ZX, ZS = 24, 10, 6, 5          # grid side; zoom window row, col, size

full = ImageOps.fit(Image.open(CAT).convert("RGB"), (480, 480), Image.LANCZOS,
                    centering=(0.5, 0.38))
px   = np.asarray(full.resize((N, N), Image.LANCZOS))            # (N, N, 3) uint8
lum  = 0.299 * px[..., 0] + 0.587 * px[..., 1] + 0.114 * px[..., 2]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13.5, 7.4))

axL.imshow(px, interpolation="nearest")
axL.set_title("What you see", fontsize=13, weight="bold", loc="left", pad=12)

axR.imshow(px, interpolation="nearest", alpha=0.28)
axR.set_title("What the computer stores", fontsize=13, weight="bold", loc="left", pad=12)
for i in range(N):
    for j in range(N):
        axR.text(j, i, f"{lum[i, j]:.0f}", ha="center", va="center",
                 fontsize=5.0, family="DejaVu Sans Mono", color=INK)

for ax, grid in ((axL, "#ffffff"), (axR, "#c9c8c3")):
    frame(ax)
    for k in range(N + 1):
        ax.axhline(k - .5, color=grid, lw=.45, alpha=.55)
        ax.axvline(k - .5, color=grid, lw=.45, alpha=.55)
    ax.add_patch(Rectangle((ZX - .5, ZY - .5), ZS, ZS, fill=False, ec=RED, lw=2.2, zorder=5))

fig.text(.5, .045, f"{N} × {N} pixels · the same data on both sides · "
                   "red box = the patch magnified below",
         ha="center", fontsize=10, color=INK_SOFT)
fig.tight_layout(rect=[0, .07, 1, 1])
plt.show()

In [ ]:
patch = px[ZY:ZY + ZS, ZX:ZX + ZS]                    # (5, 5, 3)

def number_grid(ax, plane, colour, ox=0, oy=0, fontsize=9, z=0):
    """Draw one matrix as coloured cells with the value written in each."""
    for i in range(plane.shape[0]):
        for j in range(plane.shape[1]):
            v = plane[i, j] / 255
            ax.add_patch(Rectangle((ox + j, oy - i), 1, 1, facecolor=tint(colour, v),
                                   edgecolor="w", lw=1.0, zorder=z))
            ax.text(ox + j + .5, oy - i + .5, f"{plane[i, j]}", ha="center", va="center",
                    fontsize=fontsize, family="DejaVu Sans Mono", zorder=z + 1,
                    color="w" if v > .62 else INK)
    ax.add_patch(Rectangle((ox, oy - plane.shape[0] + 1), plane.shape[1], plane.shape[0],
                           fill=False, ec=colour, lw=2.2, zorder=z + 2))

fig = plt.figure(figsize=(13.5, 7.8))
gs  = fig.add_gridspec(2, 3, height_ratios=[1, .95], hspace=.34, wspace=.18,
                       left=.05, right=.96, top=.90, bottom=.09)

# --- the patch, magnified ---------------------------------------------------
ax = fig.add_subplot(gs[0, 0])
ax.imshow(patch, interpolation="nearest")
frame(ax)
ax.set_title(f"{ZS} × {ZS} patch, magnified", fontsize=12, weight="bold", loc="left", pad=10)
for k in range(ZS + 1):
    ax.axhline(k - .5, color="w", lw=.9, alpha=.55)
    ax.axvline(k - .5, color="w", lw=.9, alpha=.55)
r, c = 1, 1
ax.add_patch(Rectangle((c - .5, r - .5), 1, 1, fill=False, ec=INK, lw=2.5, zorder=5))
ax.text(.5, -.10, f"boxed pixel = ({patch[r, c, 0]}, {patch[r, c, 1]}, {patch[r, c, 2]})",
        transform=ax.transAxes, ha="center", fontsize=11, weight="bold",
        family="DejaVu Sans Mono")
ax.text(.5, -.185, "one pixel = three numbers", transform=ax.transAxes,
        ha="center", fontsize=9.5, color=INK_SOFT)

# --- the same patch, as three stacked matrices ------------------------------
ax = fig.add_subplot(gs[0, 1:])
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("The same patch as the computer holds it: three matrices, stacked",
             fontsize=12, weight="bold", loc="left", pad=10)
DX, DY = 2.0, 1.0
for depth, (name, colour) in list(enumerate(CHANNELS))[::-1]:      # B, then G, then R on top
    ox, oy = depth * DX, depth * DY
    number_grid(ax, patch[..., depth], colour, ox, oy, fontsize=9, z=10 * (2 - depth))
    ax.text(ox - .45, oy + 1.1, name, fontsize=15, weight="bold", color=colour,
            ha="center", va="center", zorder=99)
ax.set_xlim(-1.1, 2 * DX + ZS + 3.2)
ax.set_ylim(-ZS + .3, 2 * DY + 2.4)
ax.text(2 * DX + ZS + 3.0, -ZS + 1.1, f"shape\n(3, {ZS}, {ZS})\n= {3 * ZS * ZS} numbers",
        fontsize=10, color=INK_SOFT, ha="right", va="bottom", linespacing=1.5)

# --- the three matrices, flat -----------------------------------------------
for k, (name, colour) in enumerate(CHANNELS):
    ax = fig.add_subplot(gs[1, k])
    ax.set_aspect("equal"); ax.axis("off")
    number_grid(ax, patch[..., k], colour, fontsize=11)
    ax.set_xlim(-.35, ZS + .35); ax.set_ylim(-ZS + .55, 1.75)
    ax.text(0, 1.15, f"{name} channel", fontsize=12, weight="bold", color=colour)

fig.text(.5, .022, "Zoom in far enough and there is nothing but integers 0–255. "
                   "No edges, no fur, no cat.",
         ha="center", fontsize=10.5, color=INK_SOFT)
plt.show()

## 2. Images as tensors

**Tensor**

- An *n*-dimensional array of numbers
- **rank** = how many axes; **shape** = the size of each axis

| Rank | Name | Example | Shape |
|:--|:--|:--|:--|
| 0 | scalar | a loss value | `()` |
| 1 | vector | two class scores | `(2,)` |
| 2 | matrix | one colour channel | `(176, 176)` |
| 3 | image | an RGB photo | `(3, 176, 176)` |
| 4 | batch | 32 RGB photos | `(32, 3, 176, 176)` |

**PyTorch convention**

- Channels first: `(C, H, W)`, then a batch axis in front: `(N, C, H, W)`
- `ToTensor()` rescales 0–255 integers to 0.0–1.0 floats
- `Normalize()` recentres each channel to roughly mean 0, std 1 — the range the pretrained weights expect

**Why not a plain NumPy array**

- `.to(device)` moves it to the GPU
- Every operation is recorded, so gradients come for free
- Same indexing, same broadcasting rules

In [ ]:
to_tensor = transforms.ToTensor()
normalise = transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet statistics
                                 std=[0.229, 0.224, 0.225])

x = to_tensor(full)

print("PIL image  ->  tensor")
print(f"  type      {type(x).__name__}")
print(f"  shape     {tuple(x.shape)}        (channels, height, width)")
print(f"  dtype     {x.dtype}")
print(f"  range     {x.min():.3f} .. {x.max():.3f}")
print(f"  elements  {x.numel():,}")

batched = x.unsqueeze(0)
print(f"\nwith a batch axis   {tuple(batched.shape)}   (batch, channels, height, width)")

small = to_tensor(full.resize((N, N), Image.LANCZOS))
i, j = ZY + 1, ZX + 1
print(f"\nthe boxed pixel, straight out of the tensor")
print(f"  small[:, {i}, {j}] as floats  {[round(v, 3) for v in small[:, i, j].tolist()]}")
print(f"  small[:, {i}, {j}] as 0-255   {(small[:, i, j] * 255).round().int().tolist()}")

xn = normalise(x)
print(f"\nbefore Normalize   mean {x.mean():+.3f}   std {x.std():.3f}   range {x.min():+.2f} .. {x.max():+.2f}")
print(f"after  Normalize   mean {xn.mean():+.3f}   std {xn.std():.3f}   range {xn.min():+.2f} .. {xn.max():+.2f}")

## 3. One convolutional layer

**A fully connected neuron**

- One weight for every input pixel
- A 176 × 176 × 3 image needs 92,928 weights — for one neuron
- Position is absolute: the same cat shifted three pixels is an unrelated pattern

**A convolutional neuron**

- One small kernel, e.g. 3 × 3 × 3 = 27 weights and one bias
- The same kernel is applied at every position
- Output: a **feature map** — one number per position

**Why it works on pictures**

- Shift the cat, and the feature map shifts with it
- A useful edge detector is useful everywhere in the frame
- 27 weights instead of 92,928, and each is trained on every patch of every image

**The layer, end to end**

- Multiply and sum → add bias → ReLU → pool
- Many kernels per layer → many feature maps
- Early layers find edges; deep layers find eyes, ears, muzzles

In [ ]:
im12 = np.asarray(full.resize((12, 12), Image.LANCZOS)) / 255.

fig, ax = plt.subplots(figsize=(14.5, 5.4))
ax.set_xlim(0, 132); ax.set_ylim(-2, 46); ax.set_aspect("equal"); ax.axis("off")

def flow(x0, x1, y, label=None, sub=None):
    ax.add_patch(FancyArrowPatch((x0, y), (x1, y), arrowstyle="-|>", mutation_scale=16,
                                 lw=1.6, color=INK_SOFT, shrinkA=0, shrinkB=0, zorder=30))
    if label:
        ax.text((x0 + x1) / 2, y + 1.8, label, ha="center", fontsize=9.5, color=INK)
    if sub:
        ax.text((x0 + x1) / 2, y - 4.4, sub, ha="center", fontsize=8.5, color=INK_SOFT)

BASE_Y, S, dx, dy = 11., 18., 2.8, 2.2

# --- the input volume -------------------------------------------------------
for d in (2, 1, 0):
    x0, y0 = 6 + d * dx, BASE_Y + d * dy
    if d == 0:
        ax.imshow(im12, extent=(x0, x0 + S, y0, y0 + S), zorder=10, interpolation="nearest")
    ax.add_patch(Rectangle((x0, y0), S, S, facecolor="none" if d == 0 else SURFACE,
                           ec=CHANNELS[d][1], lw=2.0, zorder=11 if d == 0 else 9 - d))
    ax.text(x0 + S - 1.0, y0 + S + 1.0, CHANNELS[d][0], fontsize=10, weight="bold",
            color=CHANNELS[d][1], ha="center", va="bottom", zorder=12)
VR = 6 + 2 * dx + S
ax.text(6, BASE_Y - 3.2, "input image", fontsize=11.5, weight="bold")
ax.text(6, BASE_Y - 6.4, "3 × H × W", fontsize=9, color=INK_SOFT)

rf = 3 * (S / 12)
rx, ry = 6 + 4 * (S / 12), BASE_Y + 5 * (S / 12)
ax.add_patch(Rectangle((rx, ry), rf, rf, fill=False, ec=INK, lw=2.6, zorder=20))
ax.annotate("", xy=(VR + 1.0, ry + rf / 2), xytext=(rx + rf, ry + rf / 2), zorder=25,
            arrowprops=dict(arrowstyle="-", color=INK, lw=1.4, ls=(0, (3, 2))))

# --- the node ---------------------------------------------------------------
nx, ny = 62, BASE_Y + S / 2
flow(VR + 1.5, nx - 7.2, ny, "one 3 × 3 window", "3 × 3 × 3 = 27 numbers")
ax.add_patch(Circle((nx, ny), 7.0, facecolor="#eef4fd", ec=BLUE, lw=2.2, zorder=10))
ax.text(nx, ny, r"$\sum K \cdot I + b$", ha="center", va="center", fontsize=13, zorder=11)
ax.text(nx, BASE_Y - 3.2, "the node", fontsize=11.5, weight="bold", ha="center")
ax.text(nx, BASE_Y - 7.6, "27 weights + 1 bias,\nreused at every position",
        ha="center", fontsize=8.5, color=INK_SOFT, linespacing=1.5)

# --- the feature map --------------------------------------------------------
fm = np.array([[.15,.30,.55,.80,.62,.35,.20,.10,.22,.40],
               [.25,.48,.75,.95,.78,.50,.28,.18,.35,.55],
               [.40,.66,.88,.72,.55,.72,.46,.30,.50,.70],
               [.55,.80,.62,.40,.35,.85,.66,.45,.62,.82],
               [.42,.60,.45,.28,.50,.70,.55,.38,.48,.66],
               [.30,.42,.32,.22,.40,.52,.40,.26,.34,.48],
               [.20,.28,.24,.18,.28,.36,.28,.18,.24,.34],
               [.14,.20,.18,.14,.20,.25,.20,.13,.17,.24],
               [.24,.34,.30,.22,.32,.40,.32,.21,.28,.38],
               [.36,.50,.44,.32,.46,.58,.46,.30,.40,.54]])
FS = 17.; FX, FY = 87, ny - FS / 2
flow(nx + 8.5, FX - 2.0, ny, "ReLU", "max(0, x)")
ax.imshow(fm, extent=(FX, FX + FS, FY, FY + FS), cmap="Blues", vmin=0, vmax=1,
          interpolation="nearest", zorder=10)
ax.add_patch(Rectangle((FX, FY), FS, FS, fill=False, ec=INK_SOFT, lw=1.4, zorder=11))
cw = FS / 10
ax.add_patch(Rectangle((FX + 3 * cw, FY + FS - 4 * cw), cw, cw, fill=False,
                       ec=INK, lw=2.4, zorder=20))
ax.text(FX, BASE_Y - 3.2, "feature map", fontsize=11.5, weight="bold")
ax.text(FX, BASE_Y - 6.4, "one number per position", fontsize=9, color=INK_SOFT)
ax.text(FX + FS / 2, FY + FS + 2.0, "the same node, slid over every position",
        ha="center", fontsize=9.5, color=INK_SOFT, style="italic")

# --- pooling ----------------------------------------------------------------
PS = 12.; PX, PY = 116, ny - PS / 2
flow(FX + FS + 2.0, PX - 2.0, ny, "max-pool", "2 × 2")
ax.imshow(fm.reshape(5, 2, 5, 2).max(axis=(1, 3)), extent=(PX, PX + PS, PY, PY + PS),
          cmap="Blues", vmin=0, vmax=1, interpolation="nearest", zorder=10)
ax.add_patch(Rectangle((PX, PY), PS, PS, fill=False, ec=INK_SOFT, lw=1.4, zorder=11))
ax.text(PX, BASE_Y - 3.2, "pooled map", fontsize=11.5, weight="bold")
ax.text(PX, BASE_Y - 6.4, "on to the next layer", fontsize=9, color=INK_SOFT)

ax.text(0, 44, "One convolutional layer", fontsize=14.5, weight="bold")
ax.text(0, 40.4, "The picture goes in on the left. One small node does the same arithmetic at "
                 "every position and writes down what it found.",
        fontsize=10, color=INK_SOFT)
plt.show()

## 4. The convolution equation

$$
S(i,\,j) \;=\; b \;+\; \sum_{c=0}^{C_{\text{in}}-1} \;\sum_{u=0}^{k_h-1} \;\sum_{v=0}^{k_w-1} \; K(c,\,u,\,v) \;\cdot\; I\bigl(c,\; i\,s + u - p,\; j\,s + v - p\bigr)
$$

**Every term**

| Symbol | Is | Meaning |
|:--|:--|:--|
| $I$ | tensor $(C_{\text{in}}, H, W)$ | the input — the image, or the previous layer's feature maps |
| $K$ | tensor $(C_{\text{in}}, k_h, k_w)$ | the **kernel**: the weights being learned, one set per output channel |
| $b$ | scalar | the **bias**, one per output channel — shifts the whole map up or down |
| $S(i,j)$ | scalar | one cell of the output feature map |
| $c$ | index | which input channel — the sum runs over all of them |
| $u,\,v$ | index | position *inside* the kernel |
| $i,\,j$ | index | position in the *output* |
| $s$ | integer | **stride** — how far the window jumps between positions |
| $p$ | integer | **padding** — rows and columns of zeros added around the edge |

**Reading it aloud**

- Line up the kernel with a window of the input
- Multiply each weight by the number underneath it
- Add all $C_{\text{in}} \times k_h \times k_w$ products together, add the bias
- Write the result into one cell of $S$, then slide the window along by $s$

**Output size**

$$
H_{\text{out}} = \left\lfloor \frac{H + 2p - k_h}{s} \right\rfloor + 1
\qquad
W_{\text{out}} = \left\lfloor \frac{W + 2p - k_w}{s} \right\rfloor + 1
$$

- $p = 1$, $k = 3$, $s = 1$ keeps the size unchanged — the usual choice
- $s = 2$ halves it

**One footnote**

- Strictly, that formula is *cross-correlation*; true convolution flips the kernel first
- Every deep learning library does it unflipped, because the kernel is learned — it simply learns the flipped version if that is what helps

In [ ]:
def convolve(I, K, b=0.0, stride=1, padding=0):
    """The equation above, written out as loops.  I: (C, H, W)   K: (C, kh, kw)."""
    C, H, W = I.shape
    _, kh, kw = K.shape
    Ip = np.pad(I, ((0, 0), (padding, padding), (padding, padding)))

    H_out = (H + 2 * padding - kh) // stride + 1
    W_out = (W + 2 * padding - kw) // stride + 1
    S = np.zeros((H_out, W_out))

    for i in range(H_out):                     # position in the output
        for j in range(W_out):
            total = b                          # start from the bias
            for c in range(C):                 # every input channel
                for u in range(kh):            # every row of the kernel
                    for v in range(kw):        # every column of the kernel
                        total += K[c, u, v] * Ip[c, i * stride + u, j * stride + v]
            S[i, j] = total
    return S


rng = np.random.default_rng(SEED)
I = rng.random((3, 12, 12))
K = rng.standard_normal((3, 3, 3))
b = 0.5

mine = convolve(I, K, b, stride=1, padding=1)
torchs = F.conv2d(torch.from_numpy(I)[None], torch.from_numpy(K)[None],
                  torch.tensor([b], dtype=torch.float64), stride=1, padding=1)[0, 0].numpy()

print(f"loops written from the equation   {mine.shape}")
print(f"torch.nn.functional.conv2d        {torchs.shape}")
print(f"largest disagreement              {np.abs(mine - torchs).max():.2e}")
print(f"\noutput size check:  (12 + 2·1 - 3) // 1 + 1 = {(12 + 2 * 1 - 3) // 1 + 1}")
print(f"with stride 2:      (12 + 2·1 - 3) // 2 + 1 = {(12 + 2 * 1 - 3) // 2 + 1}")

**Kernels are not mysterious**

- A kernel is nine numbers (times the number of input channels)
- Hand-picked ones already do useful work
- Training just means *searching* for the numbers that help most

In [ ]:
KERNELS = {
    "vertical edges":   [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "horizontal edges": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    "diagonal edges":   [[-2, -1, 0], [-1, 0, 1], [0, 1, 2]],
    "Laplacian":        [[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
}

grey = np.asarray(full.convert("L"), dtype=np.float32) / 255.
grey_t = torch.from_numpy(grey)[None, None]

fig, axes = plt.subplots(1, 5, figsize=(15, 4.2))
axes[0].imshow(grey, cmap="gray")
axes[0].set_title("input\n(one channel)", fontsize=11, weight="bold", pad=10)

for ax, (name, k) in zip(axes[1:], KERNELS.items()):
    kt = torch.tensor(k, dtype=torch.float32)[None, None]
    response = F.conv2d(grey_t, kt, padding=1)[0, 0].numpy()
    limit = np.percentile(np.abs(response), 99)      # ignore a few extreme pixels
    ax.imshow(response, cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(name, fontsize=11, weight="bold", pad=10)
    ax.set_xlabel("\n".join("  ".join(f"{v:>2}" for v in row) for row in k),
                  fontsize=8.5, family="DejaVu Sans Mono", labelpad=8, linespacing=1.5)

for ax in axes:
    frame(ax)

fig.text(.5, .015, "red = positive response · white = none · blue = negative.   "
                   "Nine numbers per kernel, and each one finds a different structure.",
         ha="center", fontsize=10, color=INK_SOFT)
fig.tight_layout(rect=[0, .06, 1, 1])
plt.show()

## 5. Labels: 0 is cat, 1 is dog

**The dataset**

- Oxford-IIIT Pet — 7,349 photographs, 37 breeds
- 12 cat breeds, 25 dog breeds
- The binary target collapses breed down to species

**The label is only an index**

- `0` means cat, `1` means dog — the model never sees the words
- The download ships 37 breed labels; the `Binary` wrapper below collapses them to one bit
- Its last layer emits **two numbers**; the larger one wins
- Which index means what is a convention we choose and then stick to

| Label $y$ | Species | Target the loss compares against |
|:--|:--|:--|
| `0` | cat | `[1, 0]` |
| `1` | dog | `[0, 1]` |

**Splits**

- `trainval` is cut into a training set and a validation set
- The `test` split is never touched until section 9
- Different photos, different animals — that is the only way an accuracy number means anything

**One wrinkle**

- The dataset is roughly one third cats, two thirds dogs
- So *always guess dog* already scores about 68% — that is the number to beat, not 50%
- Per-class recall in section 9 keeps this honest

In [ ]:
IMG_SIZE = 176
CLASSES  = ["cat", "dog"]                 # index 0, index 1

CAT_BREEDS = {                            # the 12 cat breeds; the other 25 are dogs
    "Abyssinian", "Bengal", "Birman", "Bombay", "British Shorthair", "Egyptian Mau",
    "Maine Coon", "Persian", "Ragdoll", "Russian Blue", "Siamese", "Sphynx",
}


class Binary(Dataset):
    """The 37 breeds collapsed to one bit: 0 for a cat, 1 for a dog."""

    def __init__(self, split, transform):
        self.pets = OxfordIIITPet(DATA, split=split, transform=transform)
        is_cat = [breed in CAT_BREEDS for breed in self.pets.classes]
        assert sum(is_cat) == 12, "breed names did not line up"
        self.labels = np.array([0 if is_cat[y] else 1 for y in self.pets._labels])
        self.paths = list(self.pets._images)

    def __len__(self):
        return len(self.pets)

    def __getitem__(self, i):
        image, _breed = self.pets[i]
        return image, int(self.labels[i])


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    normalise,
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    normalise,
])

N_TRAIN, N_VAL = 1600, 600

augmented = Binary("trainval", train_tf)
plain     = Binary("trainval", eval_tf)
test_ds   = Binary("test", eval_tf)
order     = torch.randperm(len(augmented), generator=torch.Generator().manual_seed(SEED)).tolist()

train_ds = Subset(augmented, order[:N_TRAIN])
val_ds   = Subset(plain, order[N_TRAIN:N_TRAIN + N_VAL])

for name, idx in (("train", order[:N_TRAIN]), ("val", order[N_TRAIN:N_TRAIN + N_VAL])):
    y = augmented.labels[idx]
    print(f"{name:<6} {len(idx):>5} images    cats {(y == 0).sum():>4}    dogs {(y == 1).sum():>4}")
print(f"{'test':<6} {len(test_ds):>5} images    "
      f"cats {(test_ds.labels == 0).sum():>4}    "
      f"dogs {(test_ds.labels == 1).sum():>4}   (held out)")

In [ ]:
def show(ax, path):
    ax.imshow(ImageOps.fit(Image.open(path).convert("RGB"), (200, 200), Image.LANCZOS))
    frame(ax)

labels = augmented.labels
paths  = np.array(augmented.paths)
picks = {0: paths[labels == 0][[3, 40, 120, 260, 430]],
         1: paths[labels == 1][[5, 90, 300, 610, 880]]}

fig = plt.figure(figsize=(14, 6.2))
gs  = fig.add_gridspec(2, 6, width_ratios=[.62] + [1] * 5, wspace=.08, hspace=.22,
                       left=.03, right=.98, top=.86, bottom=.04)

for row, (y, colour) in enumerate(((0, BLUE), (1, ORANGE))):
    ax = fig.add_subplot(gs[row, 0]); ax.axis("off")
    ax.text(.52, .56, str(y), fontsize=76, weight="bold", color=colour,
            ha="center", va="center")
    ax.text(.52, .18, CLASSES[y], fontsize=15, color=colour, ha="center", va="center")
    for col, p in enumerate(picks[y]):
        ax = fig.add_subplot(gs[row, col + 1])
        show(ax, p)
        ax.set_title(f"y = {y}", fontsize=10, color=colour, weight="bold", pad=5)

fig.text(.03, .935, "Every photograph carries one integer", fontsize=14, weight="bold")
fig.text(.03, .895, "That integer is the entire supervision signal. Everything else is learned from it.",
         fontsize=10, color=INK_SOFT)
plt.show()

## 6. The model

**ResNet-18**

- 18 weight layers, ~11.7 million parameters
- Trained on ImageNet: 1.28 million photographs, 1,000 classes
- Among those 1,000 classes are dozens of cat and dog breeds, so the features are already close to this task

**Transfer learning**

- Keep the convolutional backbone — the part that turns pixels into features
- Throw away the 1,000-way head, bolt on a 2-way one
- Freeze the early layers, train `layer4` and `fc`

**Why freeze anything**

- Early filters are generic — edges, colour blobs, textures. They already work, and retraining them on 1,600 photographs would only spoil them
- Fewer gradients to compute means faster epochs
- Most of ResNet-18's parameters sit in `layer4`, so this still leaves plenty of capacity to adapt. The printout below gives the exact split

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))     # 1000 classes -> 2

for name, p in model.named_parameters():
    p.requires_grad_(name.startswith(("layer4", "fc")))       # train the last block + the head

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"trainable   {trainable:>12,}")
print(f"frozen      {frozen:>12,}")
print(f"total       {trainable + frozen:>12,}      ({trainable / (trainable + frozen):.1%} trainable)")

print(f"\n{'stage':<12}{'output shape':>20}   what it holds")
probe = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
notes = {"conv1": "64 edge-ish maps", "bn1": "rescaled per channel",
         "relu": "negatives clamped to zero", "maxpool": "downsampled 2x",
         "layer1": "low-level texture", "layer2": "parts", "layer3": "bigger parts",
         "layer4": "object-level features", "avgpool": "one number per feature map",
         "fc": "two class scores"}
with torch.no_grad():
    for name, block in model.named_children():
        probe = block(torch.flatten(probe, 1) if name == "fc" else probe)
        print(f"{name:<12}{str(tuple(probe.shape)):>20}   {notes.get(name, '')}")

## 7. Backpropagation, epochs, hyperparameters

**Backpropagation, in plain words**

- **Forward** — the image passes through every layer and comes out as two numbers, which the loss turns into one number: *how wrong was that*
- **Backward** — the chain rule runs the same path in reverse and hands every weight a gradient
- A gradient answers one question: *if I nudge this weight up a little, how much does the loss go up?*
- Every weight then steps a little in the opposite direction. That is all training is

**The chain rule is the whole trick**

$$
\frac{\partial L}{\partial w} \;=\; \frac{\partial L}{\partial z}\,\cdot\,\frac{\partial z}{\partial w}
\qquad\qquad
w \;\leftarrow\; w \,-\, \eta\,\frac{\partial L}{\partial w}
$$

- Multiply the local derivatives along the path from the weight to the loss
- Layers share those paths, so a single backward sweep reaches every weight at once
- In PyTorch this is one line: `loss.backward()`

**Epoch**

- One **epoch** = one full pass over the training set
- The set is served in **mini-batches** — 32 images at a time
- One batch = one forward, one backward, one update. That is a **step**
- steps per epoch = training images ÷ batch size — here $1600 / 32 = 50$
- More epochs means more chances to fit — and, past a point, to overfit

**The knobs you actually turn**

| Hyperparameter | Controls | Reasonable | If it is wrong |
|:--|:--|:--|:--|
| **learning rate** $\eta$ | how big a step each weight takes | `1e-4` – `1e-2` | too high: loss jumps around or blows up. too low: nothing moves |
| **batch size** | images per update | `16` – `128` | small: noisy gradients. large: more memory, often worse generalisation |
| **epochs** | passes over the data | `3` – `30` | too few: underfit. too many: val loss climbs while train loss falls |
| **optimiser** | how a gradient becomes a step | AdamW, SGD + momentum | plain SGD is slower to converge |
| **weight decay** | pulls weights toward zero | `1e-4` – `1e-2` | too high: underfits |
| **frozen depth** | how much of the backbone adapts | last block only | unfreeze everything on a small set and it memorises it |
| **augmentation** | synthetic variety | flip, crop, jitter | none: memorises the training photos |

**The one number that matters**

- Training loss falling only proves the model can memorise
- **Validation** accuracy is the honest signal — and it is measured on photos never used for a weight update

In [ ]:
fig = plt.figure(figsize=(14.5, 5.4))
gs  = fig.add_gridspec(1, 2, width_ratios=[1.9, 1], wspace=.14,
                       left=.03, right=.97, top=.87, bottom=.08)

# --- forward and backward ---------------------------------------------------
ax = fig.add_subplot(gs[0, 0]); ax.set_xlim(0, 100); ax.set_ylim(0, 46); ax.axis("off")
ax.set_title("Backpropagation", fontsize=14, weight="bold", loc="left", pad=12)

blocks = [("image", 2), ("conv 1", 19), ("conv 2", 36), ("...", 53), ("2 scores", 67)]
BW, BH, BY, GY, LX = 14, 9, 27, 15, 84

for name, x in blocks:
    ax.add_patch(FancyBboxPatch((x, BY), BW, BH, boxstyle="round,pad=0,rounding_size=1.4",
                                facecolor="#eef4fd", ec=BLUE, lw=1.6))
    ax.text(x + BW / 2, BY + BH / 2, name, ha="center", va="center", fontsize=10.5)
for _, x in blocks[:-1]:
    ax.add_patch(FancyArrowPatch((x + BW, BY + BH / 2), (x + 16.4, BY + BH / 2),
                                 arrowstyle="-|>", mutation_scale=13, lw=1.5,
                                 color=BLUE, shrinkA=0, shrinkB=0))
ax.add_patch(FancyBboxPatch((LX, BY), 13, BH, boxstyle="round,pad=0,rounding_size=1.4",
                            facecolor="#fdefe8", ec=ORANGE, lw=1.8))
ax.text(LX + 6.5, BY + BH / 2, "loss", ha="center", va="center", fontsize=11, weight="bold")
ax.add_patch(FancyArrowPatch((67 + BW, BY + BH / 2), (LX, BY + BH / 2), arrowstyle="-|>",
                             mutation_scale=13, lw=1.5, color=BLUE, shrinkA=0, shrinkB=0))
ax.text(2, BY + BH + 3.6, "forward   ·   how wrong am I?", fontsize=11.5, weight="bold", color=BLUE)
ax.text(LX + 6.5, BY + BH + 1.6, "one number", ha="center", fontsize=8.5, color=INK_SOFT)

ax.add_patch(FancyArrowPatch((LX + 6.5, BY - 1.0), (LX + 6.5, GY), arrowstyle="-",
                             lw=1.5, color=ORANGE, shrinkA=0, shrinkB=0))
ax.add_patch(FancyArrowPatch((LX + 6.5, GY), (19, GY), arrowstyle="-|>", mutation_scale=15,
                             lw=1.6, color=ORANGE, shrinkA=0, shrinkB=0))
for _, x in blocks[1:]:
    ax.plot([x + BW / 2], [GY], "o", color=ORANGE, ms=6, zorder=5,
            markeredgecolor=SURFACE, markeredgewidth=1.4)
    ax.text(x + BW / 2, GY - 6.2, r"$\dfrac{\partial L}{\partial w}$", ha="center",
            fontsize=12.5, color=ORANGE)
ax.text(2, GY + 3.4, "backward   ·   who is to blame?", fontsize=11.5, weight="bold", color=ORANGE)
ax.text(2, 3.4, "Every weight is told: \"nudge me up, and the loss moves by this much.\"\n"
                "Each one then steps a little the other way. Repeat.",
        fontsize=9.5, color=INK_SOFT, linespacing=1.7)

# --- one weight rolling downhill --------------------------------------------
ax = fig.add_subplot(gs[0, 1])
ax.set_title("One weight, stepping downhill", fontsize=12.5, weight="bold", loc="left", pad=12)

L  = lambda x: .55 * x ** 2 + .18 * x + .60
dL = lambda x: 1.10 * x + .18
w  = np.linspace(-2.5, 2.5, 400)
ax.plot(w, L(w), color="#9c9b96", lw=2.2, zorder=2)

steps = [-2.15]
for _ in range(4):
    steps.append(steps[-1] - .40 * dL(steps[-1]))
for a, c in zip(steps, steps[1:]):
    ax.annotate("", xy=(c, L(c)), xytext=(a, L(a)), zorder=4,
                arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=1.8, shrinkA=3, shrinkB=3))
ax.plot(steps[:-1], [L(s) for s in steps[:-1]], "o", color=BLUE, ms=8.5, zorder=5,
        markeredgecolor="w", markeredgewidth=1.6)
ax.plot([steps[-1]], [L(steps[-1])], "o", color=ORANGE, ms=9.5, zorder=6,
        markeredgecolor="w", markeredgewidth=1.6)

g  = dL(steps[0])
tx = np.array([steps[0] - .60, steps[0] + .10])
ax.plot(tx, L(steps[0]) + g * (tx - steps[0]), color=ORANGE, lw=2.4, zorder=3)
ax.text(-2.42, 4.30, "slope = gradient", fontsize=9.5, color=ORANGE, ha="left", va="top")
ax.text(steps[-1] + .30, L(steps[-1]) - .38, "minimum", fontsize=9.5, color=INK_SOFT, va="center")
ax.text(.44, .90, r"$w \leftarrow w - \eta\,\dfrac{\partial L}{\partial w}$",
        transform=ax.transAxes, fontsize=15, va="top")
ax.text(.44, .60, r"$\eta$  =  learning rate  =  step size", transform=ax.transAxes,
        fontsize=9.5, color=INK_SOFT, va="top")

ax.set_xlim(-2.5, 2.5); ax.set_ylim(0, 4.6)
ax.set_xlabel("weight", fontsize=10); ax.set_ylabel("loss", fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 8. The training loop

**Every line maps onto section 7**

| Line | What it is |
|:--|:--|
| `logits = model(images)` | the forward pass |
| `loss = criterion(logits, labels)` | how wrong, as one number |
| `loss.backward()` | the backward pass — fills in `.grad` on every trainable weight |
| `optimiser.step()` | $w \leftarrow w - \eta\,\partial L/\partial w$ |
| `optimiser.zero_grad()` | gradients accumulate by default; clear them first |

**Cross-entropy loss**

$$
p_k = \frac{e^{z_k}}{e^{z_0} + e^{z_1}}
\qquad\qquad
L = -\log p_y
$$

- $z$ — the two raw scores the network emits, called **logits**
- $p$ — those scores squashed into probabilities that sum to 1, by **softmax**
- $L$ — the negative log of the probability given to the *correct* class
- Confident and right → $L$ near 0. Confident and wrong → $L$ large

In [ ]:
LR           = 3e-4
BATCH_SIZE   = 32
EPOCHS       = 4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 0        # decoding costs ~3 ms an image; worker processes are not worth it here

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **loader_args)
val_loader   = DataLoader(val_ds, shuffle=False, **loader_args)

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=LR, weight_decay=WEIGHT_DECAY)

print(f"{len(train_ds)} training images  /  batch size {BATCH_SIZE}  "
      f"=  {len(train_loader)} steps per epoch")
print(f"{EPOCHS} epochs  =  {EPOCHS * len(train_loader)} weight updates in total")

In [ ]:
def run_epoch(loader, *, train):
    model.train(train)
    total_loss = correct = seen = 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            logits = model(images)                     # forward
            loss   = criterion(logits, labels)         # how wrong

            if train:
                optimiser.zero_grad()
                loss.backward()                        # backward
                optimiser.step()                       # nudge the weights

            total_loss += loss.item() * labels.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            seen       += labels.size(0)

    return total_loss / seen, correct / seen


history = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}

def record(label, t0, train_stats, val_stats):
    for k, v in zip(history, (*train_stats, *val_stats)):
        history[k].append(v)
    print(f"{label:<12}{train_stats[0]:>10.4f}{train_stats[1]:>11.1%}"
          f"{val_stats[0]:>11.4f}{val_stats[1]:>10.1%}{time.time() - t0:>9.1f}s")

print(f"{'epoch':<12}{'train loss':>10}{'train acc':>11}{'val loss':>11}{'val acc':>10}{'time':>10}")
print("-" * 64)

t0 = time.time()                                   # epoch 0: the head is still random
record("0 (before)", t0, run_epoch(train_loader, train=False),
                          run_epoch(val_loader, train=False))

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    record(str(epoch), t0, run_epoch(train_loader, train=True),
                            run_epoch(val_loader, train=False))

print("-" * 64)
print(f"validation accuracy   {history['val_acc'][0]:.1%} before training  ->  "
      f"{max(history['val_acc']):.1%} at best")

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(13, 4.4))
epochs = range(0, EPOCHS + 1)

for ax, key, title, ylabel in ((ax_loss, "loss", "Loss", "cross-entropy"),
                               (ax_acc,  "acc",  "Accuracy", "correct")):
    for split, colour in (("train", BLUE), ("val", ORANGE)):
        y = history[f"{split}_{key}"]
        ax.plot(epochs, y, color=colour, lw=2.0, marker="o", ms=7,
                markeredgecolor=SURFACE, markeredgewidth=1.6, label=split, zorder=3)
        ax.annotate(f"{y[-1]:.3f}" if key == "loss" else f"{y[-1]:.1%}",
                    xy=(epochs[-1], y[-1]),
                    xytext=(7, 9 if split == "train" else -9),   # keep the two apart
                    textcoords="offset points",
                    color=colour, fontsize=10, weight="bold", va="center")
    ax.set_title(title, fontsize=12.5, weight="bold", loc="left", pad=10)
    ax.set_xlabel("epoch   (0 = before any training)"); ax.set_ylabel(ylabel)
    ax.set_xticks(list(epochs))
    ax.set_xlim(-.18, EPOCHS + .55)
    ax.grid(axis="y", color="#e8e7e2", lw=.9)
    ax.set_axisbelow(True)
    ax.legend(loc="center right", fontsize=10)

ax_acc.set_ylim(0, 1.03)
ax_acc.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
fig.tight_layout()
plt.show()

## 9. Predicting on an unseen cat

**The rules**

- This photograph comes from the `test` split — no weight was ever updated on it
- Different animal, different photographer, different room

**What comes out**

- Two **logits** — raw, unbounded scores. Only their *difference* matters
- **Softmax** turns them into two probabilities that sum to 1
- `argmax` picks the winner; the probability is the model's confidence

**And what is inside**

- Every intermediate tensor is readable — hooks let us copy them out mid-forward
- `conv1` holds 64 feature maps of edges and blobs; after `relu`, roughly half of every map is exactly zero
- `avgpool` holds a 512-number summary of the whole photograph. Those 512 numbers are what the final layer actually classifies

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
STD  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

def as_image(t):
    """Undo Normalize so a tensor can be looked at."""
    return (t.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

UNSEEN = int(np.flatnonzero(test_ds.labels == 0)[7])       # a cat the model has never seen

image, label = test_ds[UNSEEN]
model.eval()
with torch.no_grad():
    logits = model(image[None].to(DEVICE))[0].cpu()
probs = logits.softmax(0)

print(f"file        {test_ds.paths[UNSEEN].name}")
print(f"true label  {label}  ({CLASSES[label]})\n")
print(f"{'class':<8}{'logit z':>10}{'exp(z)':>14}{'probability':>14}")
print("-" * 46)
for k, name in enumerate(CLASSES):
    print(f"{name:<8}{logits[k]:>+10.4f}{logits[k].exp():>14.4f}{probs[k]:>14.6f}")
print("-" * 46)
print(f"{'sum':<8}{'':>10}{logits.exp().sum():>14.4f}{probs.sum():>14.6f}\n")

pred   = int(probs.argmax())
margin = (logits[pred] - logits[1 - pred]).item()

print(f"prediction   {pred}  ({CLASSES[pred]})")
print(f"margin       {margin:+.4f} logits   ->   odds {np.exp(margin):,.0f} : 1")
print(f"confidence   {probs[pred]:.4%}")
print(f"loss         {F.cross_entropy(logits[None], torch.tensor([label])):.6f}")
print(f"verdict      {'correct' if pred == label else 'WRONG'}")

In [ ]:
fig = plt.figure(figsize=(12.5, 4.8))
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 1.5], wspace=.12,
                       left=.04, right=.96, top=.82, bottom=.10)

ax = fig.add_subplot(gs[0, 0])
ax.imshow(as_image(image))
frame(ax)
ax.set_title(f"unseen photograph  ·  true label {label}", fontsize=11, loc="left", pad=8)

def as_pct(p):
    """Keep the digits that matter when the model is very sure."""
    p = float(p)
    return f"{p:.4%}" if p > .9995 or p < .0005 else f"{p:.2%}"

ax = fig.add_subplot(gs[0, 1])
colours = [BLUE, ORANGE]
ax.barh([1, 0], [probs[0].item(), probs[1].item()], height=.34,
        color=colours, zorder=3)
for y, k in ((1, 0), (0, 1)):
    ax.text(probs[k] + .015, y, as_pct(probs[k]), va="center", fontsize=13,
            weight="bold", color=INK)
    ax.text(-.02, y, f"{k}  {CLASSES[k]}", va="center", ha="right", fontsize=12,
            color=colours[k], weight="bold")
ax.set_xlim(0, 1.28); ax.set_ylim(-.5, 1.5)
ax.set_yticks([])
ax.set_xticks([0, .25, .5, .75, 1])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("softmax probability")
ax.grid(axis="x", color="#e8e7e2", lw=.9)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.set_title(f"the model says: {CLASSES[pred]}", fontsize=14, weight="bold", loc="left", pad=8)
plt.show()

In [ ]:
activations = {}

def grab(name):
    def hook(_module, _inputs, output):
        activations[name] = output.detach().cpu()
    return hook

handles = [getattr(model, n).register_forward_hook(grab(n))
           for n in ("conv1", "relu", "layer1", "layer2", "layer3", "layer4", "avgpool")]

with torch.no_grad():
    model(image[None].to(DEVICE))

for h in handles:
    h.remove()

print(f"{'layer':<10}{'shape':>22}{'mean':>10}{'max':>10}{'% zero':>10}")
print("-" * 62)
for name, a in activations.items():
    zeros = (a == 0).float().mean()
    print(f"{name:<10}{str(tuple(a.shape)):>22}{a.mean():>10.3f}{a.max():>10.3f}{zeros:>9.0%}")

embedding = activations["avgpool"].flatten()
print(f"\nthe 512 numbers the classifier actually sees")
print(f"  shape {tuple(embedding.shape)}   mean {embedding.mean():.3f}   "
      f"max {embedding.max():.3f}   nonzero {(embedding > 0).sum().item()}/512")
print(f"  strongest features (index: value)  "
      + "   ".join(f"{i}: {embedding[i]:.2f}" for i in embedding.topk(5).indices.tolist()))

In [ ]:
fig = plt.figure(figsize=(14, 6.6))
gs  = fig.add_gridspec(3, 8, height_ratios=[1, 1, .85], hspace=.30, wspace=.10,
                       left=.04, right=.97, top=.86, bottom=.10)

maps = activations["relu"][0]        # first layer, after ReLU
strongest = maps.flatten(1).max(1).values.topk(16).indices          # the liveliest 16 of 64

for k, idx in enumerate(strongest):
    ax = fig.add_subplot(gs[k // 8, k % 8])
    ax.imshow(maps[idx], cmap="Blues", vmin=0, vmax=np.percentile(maps[idx], 99.5))
    frame(ax)
    ax.set_title(f"#{idx}", fontsize=8, color=INK_SOFT, pad=3)

ax = fig.add_subplot(gs[2, :])
ax.bar(range(embedding.numel()), embedding.numpy(), width=1.0, color=BLUE, zorder=3)
ax.set_xlim(-3, 514)
ax.set_xlabel("feature index")
ax.set_ylabel("activation")
ax.set_title("avgpool  ·  the 512-number summary handed to the final layer",
             fontsize=11.5, weight="bold", loc="left", pad=8)
ax.grid(axis="y", color="#e8e7e2", lw=.9)
ax.set_axisbelow(True)

fig.text(.04, .935, "Inside the forward pass", fontsize=14, weight="bold")
fig.text(.04, .895, "Top: 16 of the 64 first-layer feature maps after ReLU, darkest where the "
                    "filter fired hardest.", fontsize=10, color=INK_SOFT)
plt.show()

**One photograph proves nothing** — the honest measure is the whole held-out split.

In [ ]:
N_TEST = 1000
test_subset = Subset(test_ds, list(range(0, len(test_ds), len(test_ds) // N_TEST))[:N_TEST])
test_loader = DataLoader(test_subset, batch_size=64, num_workers=NUM_WORKERS)

all_probs, all_true = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        all_probs.append(model(images.to(DEVICE)).softmax(1).cpu())
        all_true.append(labels)

all_probs = torch.cat(all_probs)
all_true  = torch.cat(all_true)
all_pred  = all_probs.argmax(1)

confusion = np.zeros((2, 2), dtype=int)
for t, p in zip(all_true.tolist(), all_pred.tolist()):
    confusion[t, p] += 1

print(f"held-out images   {len(all_true):,}")
print(f"accuracy          {(all_pred == all_true).float().mean():.2%}")
print(f"mean confidence   {all_probs.max(1).values.mean():.2%}\n")

print(f"{'':<14}{'said cat':>12}{'said dog':>12}   recall")
for k, name in enumerate(CLASSES):
    row = confusion[k]
    print(f"{'was ' + name:<14}{row[0]:>12,}{row[1]:>12,}   {row[k] / row.sum():>6.1%}")

In [ ]:
fig = plt.figure(figsize=(13.5, 4.0))
gs  = fig.add_gridspec(1, 2, width_ratios=[.8, 3.0], wspace=.20,
                       left=.05, right=.97, top=.80, bottom=.14)

# --- confusion matrix -------------------------------------------------------
ax = fig.add_subplot(gs[0, 0])
ax.imshow(confusion, cmap="Blues", vmin=0, vmax=confusion.max())
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{confusion[i, j]:,}", ha="center", va="center", fontsize=15,
                weight="bold", color="w" if confusion[i, j] > confusion.max() * .55 else INK)
ax.set_xticks([0, 1], [f"said {c}" for c in CLASSES])
ax.set_yticks([0, 1], [f"was {c}" for c in CLASSES])
ax.set_title("Confusion matrix", fontsize=12.5, weight="bold", loc="left", pad=10)

# --- least confident predictions --------------------------------------------
confidence = all_probs.max(1).values
hardest = confidence.topk(6, largest=False).indices

for k, pos in enumerate(hardest):
    ax = fig.add_subplot(gs[0, 1].subgridspec(1, 6, wspace=.08)[k])
    img, true_y = test_subset[int(pos)]
    ax.imshow(as_image(img))
    right = int(all_pred[pos]) == true_y
    frame(ax, AQUA if right else RED, 2.4)
    ax.set_xlabel(f"{CLASSES[int(all_pred[pos])]}  {confidence[pos]:.0%}",
                  fontsize=9, color=AQUA if right else RED, labelpad=5)
    if k == 0:
        ax.set_title("Where it hesitates  ·  the six least confident of the "
                     f"{len(all_true):,}", fontsize=12.5, weight="bold", loc="left", pad=10)
plt.show()